# AE+Ridge GPU Backtest

Walk-forward volatility forecasting with Autoencoder + Ridge regression on GPU.

**Requires GPU runtime** — go to Runtime > Change runtime type > T4 GPU.

In [ ]:
# ── Setup: clone repo, install deps, verify GPU ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)

!pip install -q torch transformers numpy pandas scikit-learn tqdm pyarrow

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected — switch to a GPU runtime")

In [ ]:
# ── Parameters ──
DATA_PATH = "all30min"
HORIZON = 1
EPOCHS = 50
BATCH_SIZE = 2
LEARNING_RATE = 1e-3
TRAIN_WINDOW = 24000
N_COMPONENTS = 5
HIDDEN_DIM = 0  # 0 = auto
ALPHA_RECON = 0.5
ALPHA_RIDGE = 1.0
MAX_LAG = 100

In [ ]:
# ── Load + transform + raw lag features ──
import numpy as np
import pandas as pd
from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH)
print(f"Loaded: {df.shape}")

adj_rv, baseline = robust_transform(
    df, "RV", use_diurnal=True, winsor_window=240, is_target=True,
)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

# Generate raw lag features (consecutive shifts)
adj_rv_arr = df["adj_RV"].values.astype(np.float64)
n = len(adj_rv_arr)
X_lags = np.full((n, MAX_LAG), np.nan, dtype=np.float64)
for lag in range(1, MAX_LAG + 1):
    X_lags[lag:, lag - 1] = adj_rv_arr[:-lag]

# Drop NaN burn-in rows
valid_mask = ~np.isnan(X_lags).any(axis=1)
first_valid = np.argmax(valid_mask)
X = X_lags[first_valid:]
y = adj_rv_arr[first_valid:]
dates = df["t"].iloc[first_valid:].reset_index(drop=True)
baselines = df["baseline"].values.astype(np.float64)[first_valid:]

print(f"Feature matrix shape: {X.shape}")
print(f"adj_RV range: [{y.min():.4f}, {y.max():.4f}]")
print(f"Total samples: {len(y):,}")

In [ ]:
# ── AE architecture ──
import torch
import torch.nn as nn


class LagAutoEncoder(nn.Module):
    def __init__(self, n_features, n_components=5, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = max(n_components, n_features // 2)
        self.encoder = nn.Sequential(
            nn.Linear(n_features, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, n_components))
        self.decoder = nn.Sequential(
            nn.Linear(n_components, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, n_features))
        self.head = nn.Linear(n_components, 1)

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        pred_rv = self.head(z).squeeze(-1)
        return reconstructed, z, pred_rv

    def encode(self, x):
        with torch.no_grad():
            return self.encoder(x)


# Show model architecture
n_features = X.shape[1]
h_dim = max(N_COMPONENTS, n_features // 2) if HIDDEN_DIM == 0 else HIDDEN_DIM
model = LagAutoEncoder(n_features, N_COMPONENTS, h_dim)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Input features:       {n_features}")
print(f"Latent components:    {N_COMPONENTS}")
print(f"Hidden dim:           {h_dim}")
print(f"Total parameters:     {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")
print()
print(f"Hybrid loss: {ALPHA_RECON} * MSE(recon, x) + {1-ALPHA_RECON} * MSE(pred, y)")
print(f"Ridge alpha: {ALPHA_RIDGE}")
print(model)

In [ ]:
# ── GPU backtest ──
import time
import gc
from src.dl_ae_ridge import (
    DEFAULT_AE_CONFIG,
    run_ae_ridge_backtest,
    apply_duan_smearing,
    apply_horizon_shift,
)

config = {
    "train_window": TRAIN_WINDOW,
    "gpu_count": 1,
    "model": {
        "n_features": n_features,
        "n_components": N_COMPONENTS,
        "hidden_dim": HIDDEN_DIM,
        "alpha_recon": ALPHA_RECON,
        "alpha_ridge": ALPHA_RIDGE,
    },
    "train": {
        "num_epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
    },
}

# Apply horizon shift
X_bt, y_bt, dates_bt, baselines_bt = apply_horizon_shift(
    X, y,
    dates,
    baselines,
    HORIZON,
)

t0 = time.time()
preds = run_ae_ridge_backtest(X_bt, y_bt, config)
elapsed = time.time() - t0
print(f"Backtest done in {elapsed:.1f}s, {len(preds)} predictions")

# Duan smearing
num_windows = len(preds)
y_oos = y_bt[TRAIN_WINDOW:TRAIN_WINDOW + num_windows]
dates_oos = dates_bt.iloc[TRAIN_WINDOW:TRAIN_WINDOW + num_windows].values
baselines_oos = baselines_bt[TRAIN_WINDOW:TRAIN_WINDOW + num_windows]

pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

results = pd.DataFrame({
    "date": dates_oos,
    "horizon": HORIZON,
    "true_adj": y_oos,
    "pred_adj": preds,
    "true_raw": true_raw,
    "pred_raw": pred_raw,
})
print(results.describe())
results.head(10)

In [ ]:
# ── Quick eval ──
from src.evaluation import calculate_metrics

metrics = calculate_metrics(results["true_raw"].values, results["pred_raw"].values)
print("Out-of-sample metrics (raw scale):")
for k, v in metrics.items():
    print(f"  {k}: {v:.6f}")

In [ ]:
%%writefile ../src/dl_ae_ridge.py
"""AE+Ridge GPU backtest executor for volatility forecasting.

Self-contained CLI: load -> transform -> AE encode -> Ridge predict -> save chunk CSV.
No imports from core/ or projects/.
"""

import argparse
import csv
import gc
import json
import logging
import math
import os
import tempfile
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.multiprocessing as mp
from torch.func import vmap, functional_call

from src.loading import load_raw_data
from src.transforms import robust_transform

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

# ── Constants ────────────────────────────────────────────────────────────
PERIODS_PER_DAY = 48

DEFAULT_AE_CONFIG = {
    "train_window": 24000,
    "gpu_count": 1,
    "model": {
        "n_features": 0,
        "n_components": 5,
        "hidden_dim": 0,
        "alpha_recon": 0.5,
        "alpha_ridge": 1.0,
    },
    "train": {
        "num_epochs": 50,
        "learning_rate": 1e-3,
        "batch_size": 2,
    },
}


# ── LagAutoEncoder Model ────────────────────────────────────────────────


class LagAutoEncoder(nn.Module):
    def __init__(self, n_features, n_components=5, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = max(n_components, n_features // 2)
        self.encoder = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_components),
        )
        self.decoder = nn.Sequential(
            nn.Linear(n_components, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_features),
        )
        self.head = nn.Linear(n_components, 1)

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        pred_rv = self.head(z).squeeze(-1)
        return reconstructed, z, pred_rv

    def encode(self, x):
        with torch.no_grad():
            return self.encoder(x)


# ── Raw lag feature generation ───────────────────────────────────────────


def generate_raw_lags(series, max_lag=100):
    n = len(series)
    X = np.full((n, max_lag), np.nan, dtype=np.float64)
    for lag in range(1, max_lag + 1):
        X[lag:, lag - 1] = series[:-lag]
    return X


# ── Instance normalization ───────────────────────────────────────────────


def instance_norm_np(X, eps=1e-8):
    mean = X.mean(axis=0, keepdims=True)
    std = X.std(axis=0, keepdims=True)
    std = np.where(std < eps, 1.0, std)
    return (X - mean) / std, mean, std


# ── AE training ──────────────────────────────────────────────────────────


def train_ae_window(model, X_train, y_train, cfg, device):
    num_epochs = cfg["train"]["num_epochs"]
    lr = cfg["train"]["learning_rate"]
    batch_size = cfg["train"]["batch_size"]
    alpha = cfg["model"]["alpha_recon"]

    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    n_samples = X_train.shape[0]

    for epoch in range(num_epochs):
        perm = torch.randperm(n_samples, device=device)
        for i in range(0, n_samples, batch_size):
            idx = perm[i : i + batch_size]
            x_batch = X_train[idx]
            y_batch = y_train[idx]

            reconstructed, z, pred_rv = model(x_batch)

            loss_recon = nn.functional.mse_loss(reconstructed, x_batch)
            loss_pred = nn.functional.mse_loss(pred_rv, y_batch)
            loss = alpha * loss_recon + (1.0 - alpha) * loss_pred

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()


# ── Ridge solve via torch.linalg ─────────────────────────────────────────


def ridge_solve(Z, y, alpha_ridge=1.0):
    k = Z.shape[1]
    ZtZ = Z.T @ Z + alpha_ridge * torch.eye(k, device=Z.device, dtype=Z.dtype)
    Zty = Z.T @ y
    w = torch.linalg.solve(ZtZ, Zty)
    return w


# ── Single window AE+Ridge prediction ───────────────────────────────────


def _predict_window(X_train, y_train, X_test, cfg, device):
    n_features = X_train.shape[1]
    n_components = cfg["model"]["n_components"]
    hidden_dim = cfg["model"]["hidden_dim"]
    if hidden_dim == 0:
        hidden_dim = None
    alpha_ridge = cfg["model"]["alpha_ridge"]

    t_mean = X_train.mean(dim=0, keepdim=True)
    t_std = X_train.std(dim=0, keepdim=True).clamp(min=1e-8)
    X_train_norm = (X_train - t_mean) / t_std
    X_test_norm = (X_test - t_mean) / t_std

    model = LagAutoEncoder(n_features, n_components, hidden_dim).to(device)
    train_ae_window(model, X_train_norm, y_train, cfg, device)

    model.eval()
    with torch.no_grad():
        Z_train = model.encode(X_train_norm)

    w = ridge_solve(Z_train, y_train, alpha_ridge)

    with torch.no_grad():
        z_test = model.encode(X_test_norm)
    pred = (z_test @ w).item()

    return pred


# ── GPU worker ───────────────────────────────────────────────────────────


def _gpu_worker(gpu_id, window_indices, X_all, y_all, config, result_dict):
    device = torch.device(f"cuda:{gpu_id}")
    torch.cuda.set_device(device)

    train_window = config["train_window"]

    X_device = torch.tensor(X_all, dtype=torch.float32, device=device)
    y_device = torch.tensor(y_all, dtype=torch.float32, device=device)

    predictions = {}
    for w_idx in window_indices:
        t_start = w_idx
        t_end = w_idx + train_window
        test_idx = t_end

        X_train = X_device[t_start:t_end]
        y_train = y_device[t_start:t_end]
        X_test = X_device[test_idx : test_idx + 1]

        pred = _predict_window(X_train, y_train, X_test, config, device)
        predictions[w_idx] = pred

        if w_idx % 100 == 0:
            torch.cuda.empty_cache()
            gc.collect()

    result_dict[gpu_id] = predictions


# ── Multi-GPU distribution ───────────────────────────────────────────────


def run_ae_ridge_backtest(X, y, config):
    gpu_count = config.get("gpu_count", 1)
    available_gpus = torch.cuda.device_count()
    gpu_count = min(gpu_count, available_gpus)

    if gpu_count == 0:
        raise RuntimeError("No CUDA GPU available")

    train_window = config["train_window"]
    num_windows = len(X) - train_window

    if gpu_count == 1:
        result_dict = {}
        _gpu_worker(0, list(range(num_windows)), X, y, config, result_dict)
        predictions = np.array([result_dict[0][i] for i in range(num_windows)])
    else:
        ctx = mp.get_context("spawn")
        manager = ctx.Manager()
        result_dict = manager.dict()

        window_splits = [[] for _ in range(gpu_count)]
        for i in range(num_windows):
            window_splits[i % gpu_count].append(i)

        processes = []
        for gpu_id in range(gpu_count):
            p = ctx.Process(
                target=_gpu_worker,
                args=(gpu_id, window_splits[gpu_id], X, y, config, result_dict),
            )
            p.start()
            processes.append(p)

        for p in processes:
            p.join()

        all_preds = {}
        for gpu_id in range(gpu_count):
            all_preds.update(result_dict[gpu_id])
        predictions = np.array([all_preds[i] for i in range(num_windows)])

    return predictions


# ── Duan smearing ────────────────────────────────────────────────────────


def apply_duan_smearing(forecasts, y_true, baselines):
    smear = np.mean((y_true - forecasts) ** 2)
    pred_raw = (forecasts**2 + smear) * baselines
    true_raw = (y_true**2) * baselines
    return pred_raw, true_raw


# ── Horizon shift ────────────────────────────────────────────────────────


def apply_horizon_shift(X, y, dates, baselines, horizon):
    if horizon <= 1:
        return X, y, dates, baselines
    shift = horizon - 1
    return (
        X[:-shift],
        y[shift:],
        dates.iloc[:-shift].reset_index(drop=True),
        baselines[shift:],
    )


# ── CLI ──────────────────────────────────────────────────────────────────


def main():
    parser = argparse.ArgumentParser(
        description="AE+Ridge GPU walk-forward backtest"
    )
    parser.add_argument("--data-path", default="all30min")
    parser.add_argument("--horizon", type=int, default=1)
    parser.add_argument("--gpu-count", type=int, default=1)
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--batch-size", type=int, default=None)
    parser.add_argument("--learning-rate", type=float, default=None)
    parser.add_argument("--chunk-id", type=int, default=0)
    parser.add_argument("--total-chunks", type=int, default=1)
    parser.add_argument("--output-file", required=True)
    parser.add_argument("--n-components", type=int, default=None)
    args = parser.parse_args()

    config = json.loads(json.dumps(DEFAULT_AE_CONFIG))
    config["gpu_count"] = args.gpu_count
    if args.epochs is not None:
        config["train"]["num_epochs"] = args.epochs
    if args.batch_size is not None:
        config["train"]["batch_size"] = args.batch_size
    if args.learning_rate is not None:
        config["train"]["learning_rate"] = args.learning_rate
    if args.n_components is not None:
        config["model"]["n_components"] = args.n_components

    logger.info(f"Loading data from {args.data_path}")
    df = load_raw_data(args.data_path)

    adj_rv, baseline = robust_transform(
        df, "RV", use_diurnal=True, winsor_window=240, is_target=True,
    )
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    adj_rv_arr = df["adj_RV"].values.astype(np.float64)
    max_lag = 100
    X_lags = generate_raw_lags(adj_rv_arr, max_lag=max_lag)

    valid_mask = ~np.isnan(X_lags).any(axis=1)
    first_valid = np.argmax(valid_mask)
    X = X_lags[first_valid:]
    y = adj_rv_arr[first_valid:]
    dates = df["t"].iloc[first_valid:].reset_index(drop=True)
    baselines_arr = df["baseline"].values.astype(np.float64)[first_valid:]

    config["model"]["n_features"] = X.shape[1]

    X, y, dates, baselines_arr = apply_horizon_shift(
        X, y, dates, baselines_arr, args.horizon
    )

    n = len(X)
    chunk_size = n // args.total_chunks
    start = args.chunk_id * chunk_size
    end = n if args.chunk_id == args.total_chunks - 1 else start + chunk_size

    X_chunk = X[start:end]
    y_chunk = y[start:end]
    dates_chunk = dates.iloc[start:end].reset_index(drop=True)
    baselines_chunk = baselines_arr[start:end]

    train_window = config["train_window"]
    if train_window >= len(X_chunk):
        raise ValueError(
            f"train_window ({train_window}) >= chunk size ({len(X_chunk)})"
        )

    t0 = time.time()
    preds = run_ae_ridge_backtest(X_chunk, y_chunk, config)
    elapsed = time.time() - t0
    logger.info(f"Backtest complete in {elapsed:.1f}s")

    num_windows = len(preds)
    y_oos = y_chunk[train_window:train_window + num_windows]
    dates_oos = dates_chunk.iloc[train_window:train_window + num_windows].values
    baselines_oos = baselines_chunk[train_window:train_window + num_windows]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame({
        "date": dates_oos,
        "horizon": args.horizon,
        "true_adj": y_oos,
        "pred_adj": preds,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    })

    os.makedirs(os.path.dirname(args.output_file) or ".", exist_ok=True)
    results.to_csv(args.output_file, index=False)
    logger.info(f"Saved {len(results)} rows -> {args.output_file}")


if __name__ == "__main__":
    main()